# 4.最終モデルを構築する
- LightGBMモデルのハイパーパラメータチューニングを学習データすべてを使い行う、その後学習データすべてを学習させて保存
- **5. Predict_test.ipynb** にて保存モデルを呼び出し、目的変数を持たないtestデータの予測を行う

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns

from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor

import lightgbm as lgb

from rdkit import Chem
from rdkit.Chem import Draw

import optuna
import joblib
import os

c:\Users\kawan\miniforge3\envs\chem_win\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## データ読み込み

### 学習データ読み込み

In [2]:
# 入力：読み込みたい記述子のタイプを選択
descriptor_type = "rdkit"  # 'rdkit' or 'mordred_2d' or 'mordred_3d' or 'fp'

dataset = pd.read_csv("../data/dataset_train.csv", index_col=0)
target = dataset[["Emission max (nm)", "Quantum yield"]]

if descriptor_type == "rdkit":
    des = pd.read_csv("outputs/descriptors/rdkit/X_train_rdkit_processed.csv", index_col=0)
elif descriptor_type == "mordred_2d":
    des = pd.read_csv(
        "outputs/descriptors/mordred_2d/X_train_mordred_2d_processed.csv", index_col=0
    )
elif descriptor_type == "mordred_3d":
    des = pd.read_csv(
        "outputs/descriptors/mordred_3d/X_train_mordred_3d_processed.csv", index_col=0
    )
elif descriptor_type == "fp":
    des = pd.read_csv("outputs/descriptors/fp/X_train_fp.csv", index_col=0)
else:
    raise ValueError(f"未知の descriptor_type: {descriptor_type}")

# 目的変数と記述子を結合
dataset_train = target.join(des, how="inner")
print(target.shape, des.shape, dataset_train.shape)

(13132, 2) (13132, 206) (13132, 208)


In [3]:
dataset_train.head()

,Emission max (nm),Quantum yield,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,HeavyAtomMolWt,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
Tag,,,,,,,,,,,,,,,,,,,,,
72,538.0,0.200,12.408729,12.408729,0.029454,-1.347366,0.284676,11.310345,645.879,639.831,...,0,0,0,0,0,0,0,0,0,0
73,534.0,0.020,12.503623,12.503623,0.021975,-1.307991,0.220631,11.310345,833.879,827.831,...,0,0,0,0,0,0,0,0,0,0
74,566.0,0.018,12.637713,12.637713,0.057996,-1.632189,0.093205,11.515152,971.659,969.643,...,0,0,0,0,0,0,0,0,0,0
75,542.0,0.600,12.408729,12.408729,0.029454,-1.347366,0.284676,11.310345,645.879,639.831,...,0,0,0,0,0,0,0,0,0,0
76,545.0,0.080,12.503623,12.503623,0.021975,-1.307991,0.220631,11.310345,833.879,827.831,...,0,0,0,0,0,0,0,0,0,0


### テストデータ読み込み

In [4]:
# 入力：読み込みたい記述子のタイプを選択
descriptor_type = "rdkit"  # 'rdkit' or 'mordred_2d' or 'mordred_3d' or 'fp'

# dataset = pd.read_csv("../data/dataset_test.csv", index_col=0)

if descriptor_type == "rdkit":
    des = pd.read_csv("outputs/descriptors/rdkit/X_test_rdkit_processed.csv", index_col=0)
elif descriptor_type == "mordred_2d":
    des = pd.read_csv(
        "outputs/descriptors/mordred_2d/X_test_mordred_2d_processed.csv", index_col=0
    )
elif descriptor_type == "mordred_3d":
    des = pd.read_csv(
        "outputs/descriptors/mordred_3d/X_test_mordred_3d_processed.csv", index_col=0
    )
elif descriptor_type == "fp":
    des = pd.read_csv("outputs/descriptors/fp/X_test_fp.csv", index_col=0)
else:
    raise ValueError(f"未知の descriptor_type: {descriptor_type}")

# 目的変数と記述子を結合
X_test = des
print(des.shape, X_test.shape)

(7104, 206) (7104, 206)


In [5]:
X_test.head()

,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,HeavyAtomMolWt,ExactMolWt,NumValenceElectrons,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
Tag,,,,,,,,,,,,,,,,,,,,,
1,11.104717,11.104717,0.018850,-0.689259,0.631799,9.928571,187.154,182.114,187.026943,68,...,0,0,0,0,0,0,0,0,0,0
2,11.084309,11.084309,0.059121,-0.717037,0.568211,9.928571,186.146,182.114,186.019667,68,...,0,0,0,0,0,0,0,0,0,0
3,14.130123,14.130123,0.240638,-0.240770,0.013057,12.784810,1061.549,968.813,1060.705709,418,...,0,0,0,0,0,0,0,0,33,0
4,13.279579,13.279579,0.064638,0.064638,0.344339,12.000000,350.422,338.326,350.064509,122,...,0,0,0,0,0,1,0,0,0,0
5,13.279579,13.279579,0.064638,0.064638,0.344339,12.000000,350.422,338.326,350.064509,122,...,0,0,0,0,0,1,0,0,0,0


## 最終モデルの作成_emission予測モデル

In [6]:
# 目的変数と説明変数
y = dataset_train["Emission max (nm)"]
X = dataset_train.drop("Emission max (nm)", axis=1)
X = X.drop("Quantum yield", axis=1)

print("X:", X.shape, "X_test:", X_test.shape)

X: (13132, 206) X_test: (7104, 206)


In [7]:
# オートスケーリング（学習データの統計量で fit）
X_scaler = StandardScaler()
autoscaled_X_train = pd.DataFrame(
    X_scaler.fit_transform(X),
    columns=X.columns,
    index=X.index,
)
autoscaled_X_test = pd.DataFrame(
    X_scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index,
)

y_scaler = StandardScaler()
autoscaled_y_train = y_scaler.fit_transform(y.values.reshape(-1, 1))

autoscaled_y_train = pd.DataFrame(
    autoscaled_y_train, index=y.index, columns=["y"]
)

In [8]:
# ハイパーパラメータチューニング
def objective(trial):
    params = {
        'objective': 'regression',
        'metric': 'rmse',  # LightGBMが内部で使う評価指標
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'num_leaves': trial.suggest_int('num_leaves', 10, 100),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 1.0, log=True),
        'random_state': 1234,
        'verbosity': -1,
        'device': 'cpu',
    }

    # 分割、CV
    kf = KFold(n_splits=5, shuffle=True, random_state=1234)

    # 評価指標
    rmse_scores = []

    for train_index, val_index in kf.split(X):
        # 訓練と検証に分類
        X_train, X_val = X.iloc[train_index, :], X.iloc[val_index, :]
        y_train, y_val = y.iloc[train_index], y.iloc[val_index]

        # 重要: スケーラーは各foldの train のみでfit（情報リークを防ぐ）
        X_scaler = StandardScaler()
        autoscaled_X_train = pd.DataFrame(
            X_scaler.fit_transform(X_train),
            columns=X_train.columns,
            index=X_train.index,
        )
        autoscaled_X_val = pd.DataFrame(
            X_scaler.transform(X_val),
            columns=X_val.columns,
            index=X_val.index,
        )

        y_scaler = StandardScaler()
        autoscaled_y_train = y_scaler.fit_transform(y_train.values.reshape(-1, 1)).ravel()

        # モデル定義・学習
        model_lgb = lgb.LGBMRegressor(**params)
        model_lgb.fit(autoscaled_X_train, autoscaled_y_train)

        # 予測（元スケールへ戻す）
        y_pred_scaled = model_lgb.predict(autoscaled_X_val)
        y_pred = y_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()

        # 性能チェック
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))

        # 結果格納
        rmse_scores.append(rmse)

    return float(np.mean(rmse_scores))  # 最適化したい評価指標を選ぶ

In [9]:
# 最適化
study = optuna.create_study(direction='minimize', study_name='regression')
study.optimize(objective, n_trials=30)

# ハイパーパラメータ・スコアの確認
print("Best trial:")
trial = study.best_trial

print(f"  RMSE: {trial.value:.4f}")
print("  Params:")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

[I 2026-05-13 23:12:58,699] A new study created in memory with name: regression
[I 2026-05-13 23:13:10,292] Trial 0 finished with value: 43.55710933584818 and parameters: {'n_estimators': 183, 'learning_rate': 0.016635604633713705, 'max_depth': 7, 'num_leaves': 59, 'subsample': 0.9064214192719704, 'colsample_bytree': 0.6585187311959646, 'reg_alpha': 0.00024588476454898747, 'reg_lambda': 8.106512765966804e-07}. Best is trial 0 with value: 43.55710933584818.
[I 2026-05-13 23:13:13,521] Trial 1 finished with value: 43.09322465200785 and parameters: {'n_estimators': 53, 'learning_rate': 0.11745413261398485, 'max_depth': 10, 'num_leaves': 24, 'subsample': 0.5571843236469, 'colsample_bytree': 0.7866856933247766, 'reg_alpha': 0.003723938552071181, 'reg_lambda': 0.005104504655516108}. Best is trial 1 with value: 43.09322465200785.
[I 2026-05-13 23:13:24,174] Trial 2 finished with value: 48.305839206545116 and parameters: {'n_estimators': 119, 'learning_rate': 0.014044292810189894, 'max_depth':

Best trial:
  RMSE: 35.5641
  Params:
    n_estimators: 263
    learning_rate: 0.0783528999772625
    max_depth: 8
    num_leaves: 48
    subsample: 0.9690490717176061
    colsample_bytree: 0.7329221703577693
    reg_alpha: 0.0008844543803883389
    reg_lambda: 0.1265973252041221


In [10]:
# optunaで最適化されたパラメータをセットし、再評価する
model_lgb_op = lgb.LGBMRegressor(**study.best_params, random_state=1234, verbosity=-1, device='cpu')

# 5分割交差検証
kf = KFold(n_splits=5, shuffle=True, random_state=1234)

# スコア保存用
rmse_scores = []
mae_scores = []
r2_scores = []

for train_index, val_index in kf.split(X):
    # 訓練と検証に分類
    X_train, X_val = X.iloc[train_index, :], X.iloc[val_index, :]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    # 標準化、DataFrameに戻す
    X_scaler = StandardScaler()
    autoscaled_X_train = pd.DataFrame(X_scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
    autoscaled_X_val = pd.DataFrame(X_scaler.transform(X_val), columns=X_val.columns, index=X_val.index)

    y_scaler = StandardScaler()
    autoscaled_y_train = pd.DataFrame(y_scaler.fit_transform(y_train.values.reshape(-1,1)), index=y_train.index, columns=['y'])
    autoscaled_y_val = pd.DataFrame(y_scaler.transform(y_val.values.reshape(-1,1)), index=y_val.index, columns=['y'])

    # 学習
    model_lgb_op.fit(autoscaled_X_train, autoscaled_y_train, eval_set=[(autoscaled_X_val, autoscaled_y_val)])
    y_pred_scaled = model_lgb_op.predict(autoscaled_X_val)
    y_pred = y_scaler.inverse_transform(y_pred_scaled.reshape(-1,1))

    # 性能チェック
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mae = mean_absolute_error(y_val, y_pred)
    r2 = r2_score(y_val, y_pred)

    rmse_scores.append(rmse)
    mae_scores.append(mae)
    r2_scores.append(r2)

    print(f'Fold RMSE : {rmse:.4f}')
    print(f'Fold MAE : {mae:.4f}')
    print(f'Fold R2 : {r2:.4f}')
    print()

print(f'平均RMSE : {np.mean(rmse_scores):.4f}')
print(f'平均MAE : {np.mean(mae_scores):.4f}')
print(f'平均R2 : {np.mean(r2_scores):.4f}')

Fold RMSE : 36.5869
Fold MAE : 24.3258
Fold R2 : 0.8489

Fold RMSE : 34.8878
Fold MAE : 23.4410
Fold R2 : 0.8568

Fold RMSE : 34.5855
Fold MAE : 23.6208
Fold R2 : 0.8661

Fold RMSE : 36.2963
Fold MAE : 24.4056
Fold R2 : 0.8541

Fold RMSE : 35.4642
Fold MAE : 24.2225
Fold R2 : 0.8646

平均RMSE : 35.5641
平均MAE : 24.0031
平均R2 : 0.8581


In [11]:
# 最終モデル構築（LGB）

# LGB本体（best_params を使用）
# 注意: lgb という変数名は lightgbm モジュール名と衝突するため使わない
lgb_emi = lgb.LGBMRegressor(**study.best_params, random_state=1234, verbosity=-1, device='cpu')

# X標準化 + LGB
x_pipe = Pipeline([
    ("x_scaler", StandardScaler()),
    ("lgb", lgb_emi),
])

# 重要: y標準化も含める（predict時は自動で元スケールに逆変換される）
model_lgb_final = TransformedTargetRegressor(
    regressor=x_pipe,
    transformer=StandardScaler()
)

# 学習
model_lgb_final.fit(X, y)

# 保存（descriptor_type を明示）
os.makedirs("models/lgb", exist_ok=True)

model_name = f"lgb_{descriptor_type}_emi"   # 例: lgb_mordred_3d_emi
save_path = f"models/lgb/artifact_{model_name}.joblib"

# 重要: 推論時に列順を再現できるよう、feature_columns も一緒に保存
artifact_lgb = {
    "model_name": model_name,
    "model_type": "LGB",
    "descriptor_type": descriptor_type,
    "model": model_lgb_final,
    "feature_columns": X.columns.tolist(),
}

joblib.dump(artifact_lgb, save_path)

print(f"saved: {save_path}")

saved: models/lgb/artifact_lgb_rdkit_emi.joblib


## 最終モデルの構築_quantum yield予測モデル

In [12]:
# 目的変数と説明変数（quantum yield）
y_qy = dataset_train["Quantum yield"]
X_qy = dataset_train.drop(columns=["Emission max (nm)", "Quantum yield"])

print("X_qy:", X_qy.shape, "X_test:", X_test.shape)

X_qy: (13132, 206) X_test: (7104, 206)


In [13]:
# ハイパーパラメータチューニング
def objective_qy(trial):
    params = {
        "objective": "regression",
        "metric": "rmse",
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "num_leaves": trial.suggest_int("num_leaves", 10, 100),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 1.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 1.0, log=True),
        "random_state": 1234,
        "verbosity": -1,
        "device": "cpu",
    }

    kf = KFold(n_splits=5, shuffle=True, random_state=1234)
    rmse_scores = []

    for train_index, val_index in kf.split(X_qy):
        X_train_qy = X_qy.iloc[train_index, :]
        X_val_qy = X_qy.iloc[val_index, :]
        y_train_qy = y_qy.iloc[train_index]
        y_val_qy = y_qy.iloc[val_index]

        X_scaler = StandardScaler()
        autoscaled_X_train_qy = pd.DataFrame(
            X_scaler.fit_transform(X_train_qy),
            columns=X_train_qy.columns,
            index=X_train_qy.index,
        )
        autoscaled_X_val_qy = pd.DataFrame(
            X_scaler.transform(X_val_qy),
            columns=X_val_qy.columns,
            index=X_val_qy.index,
        )

        y_scaler = StandardScaler()
        autoscaled_y_train_qy = y_scaler.fit_transform(
            y_train_qy.values.reshape(-1, 1)
        ).ravel()

        model_lgb_qy = lgb.LGBMRegressor(**params)
        model_lgb_qy.fit(autoscaled_X_train_qy, autoscaled_y_train_qy)

        y_pred_scaled_qy = model_lgb_qy.predict(autoscaled_X_val_qy)
        y_pred_qy = y_scaler.inverse_transform(
            y_pred_scaled_qy.reshape(-1, 1)
        ).ravel()

        rmse_scores.append(np.sqrt(mean_squared_error(y_val_qy, y_pred_qy)))

    return float(np.mean(rmse_scores))

In [14]:
# quantum yield モデルの最適化
study_qy = optuna.create_study(direction="minimize", study_name="regression_qy")
study_qy.optimize(objective_qy, n_trials=30)

print("Best trial (Quantum yield):")
trial_qy = study_qy.best_trial
print(f"  RMSE: {trial_qy.value:.4f}")
print("  Params:")
for key, value in trial_qy.params.items():
    print(f"    {key}: {value}")

[I 2026-05-13 23:25:32,253] A new study created in memory with name: regression_qy
[I 2026-05-13 23:25:33,574] Trial 0 finished with value: 0.2891170435824604 and parameters: {'n_estimators': 51, 'learning_rate': 0.010869351373575224, 'max_depth': 4, 'num_leaves': 63, 'subsample': 0.9506414980181394, 'colsample_bytree': 0.701193155406051, 'reg_alpha': 0.012002552395680143, 'reg_lambda': 0.0002319402733271654}. Best is trial 0 with value: 0.2891170435824604.
[I 2026-05-13 23:25:39,312] Trial 1 finished with value: 0.22867923479966779 and parameters: {'n_estimators': 271, 'learning_rate': 0.018319039662649157, 'max_depth': 10, 'num_leaves': 20, 'subsample': 0.5671695893548118, 'colsample_bytree': 0.8178846225055875, 'reg_alpha': 4.892623737253168e-06, 'reg_lambda': 0.0005126387601160319}. Best is trial 1 with value: 0.22867923479966779.
[I 2026-05-13 23:25:42,915] Trial 2 finished with value: 0.2054001018622948 and parameters: {'n_estimators': 72, 'learning_rate': 0.10890098840489514, 'm

Best trial (Quantum yield):
  RMSE: 0.2003
  Params:
    n_estimators: 241
    learning_rate: 0.05558077931076217
    max_depth: 10
    num_leaves: 60
    subsample: 0.8815245106964085
    colsample_bytree: 0.7680259104667736
    reg_alpha: 2.997454041751932e-05
    reg_lambda: 5.419061271910842e-06


In [15]:
# 最適化パラメータで再評価（quantum yield）
model_lgb_op_qy = lgb.LGBMRegressor(
    **study_qy.best_params, random_state=1234, verbosity=-1, device="cpu"
)

kf_qy = KFold(n_splits=5, shuffle=True, random_state=1234)
rmse_scores_qy = []
mae_scores_qy = []
r2_scores_qy = []

for train_index, val_index in kf_qy.split(X_qy):
    X_train_qy = X_qy.iloc[train_index, :]
    X_val_qy = X_qy.iloc[val_index, :]
    y_train_qy = y_qy.iloc[train_index]
    y_val_qy = y_qy.iloc[val_index]

    X_scaler = StandardScaler()
    autoscaled_X_train_qy = pd.DataFrame(
        X_scaler.fit_transform(X_train_qy),
        columns=X_train_qy.columns,
        index=X_train_qy.index,
    )
    autoscaled_X_val_qy = pd.DataFrame(
        X_scaler.transform(X_val_qy),
        columns=X_val_qy.columns,
        index=X_val_qy.index,
    )

    y_scaler = StandardScaler()
    autoscaled_y_train_qy = y_scaler.fit_transform(
        y_train_qy.values.reshape(-1, 1)
    ).ravel()

    model_lgb_op_qy.fit(autoscaled_X_train_qy, autoscaled_y_train_qy)
    y_pred_scaled_qy = model_lgb_op_qy.predict(autoscaled_X_val_qy)
    y_pred_qy = y_scaler.inverse_transform(y_pred_scaled_qy.reshape(-1, 1)).ravel()

    rmse = np.sqrt(mean_squared_error(y_val_qy, y_pred_qy))
    mae = mean_absolute_error(y_val_qy, y_pred_qy)
    r2 = r2_score(y_val_qy, y_pred_qy)

    rmse_scores_qy.append(rmse)
    mae_scores_qy.append(mae)
    r2_scores_qy.append(r2)

    print(f"Fold RMSE : {rmse:.4f}")
    print(f"Fold MAE : {mae:.4f}")
    print(f"Fold R2 : {r2:.4f}\n")

print("=== Mean CV Scores (Quantum yield) ===")
print(f"RMSE: {np.mean(rmse_scores_qy):.4f}")
print(f"MAE : {np.mean(mae_scores_qy):.4f}")
print(f"R2  : {np.mean(r2_scores_qy):.4f}")

Fold RMSE : 0.2040
Fold MAE : 0.1464
Fold R2 : 0.5600

Fold RMSE : 0.2005
Fold MAE : 0.1453
Fold R2 : 0.5870

Fold RMSE : 0.1976
Fold MAE : 0.1433
Fold R2 : 0.5852

Fold RMSE : 0.1993
Fold MAE : 0.1443
Fold R2 : 0.5688

Fold RMSE : 0.1999
Fold MAE : 0.1447
Fold R2 : 0.5743

=== Mean CV Scores (Quantum yield) ===
RMSE: 0.2003
MAE : 0.1448
R2  : 0.5751


In [16]:
# 最終モデル構築と保存（quantum yield）
lgb_qy = lgb.LGBMRegressor(
    **study_qy.best_params, random_state=1234, verbosity=-1, device="cpu"
)

x_pipe_qy = Pipeline([
    ("x_scaler", StandardScaler()),
    ("lgb", lgb_qy),
])

model_lgb_final_qy = TransformedTargetRegressor(
    regressor=x_pipe_qy,
    transformer=StandardScaler(),
)

model_lgb_final_qy.fit(X_qy, y_qy)

os.makedirs("models/lgb", exist_ok=True)

model_name_qy = f"lgb_{descriptor_type}_qy"   # 例: lgb_mordred_3d_qy
save_path_qy = f"models/lgb/artifact_{model_name_qy}.joblib"

# 重要: emission モデルと衝突しないよう、_qy サフィックスで保存
artifact_lgb_qy = {
    "model_name": model_name_qy,
    "model_type": "LGB",
    "descriptor_type": descriptor_type,
    "target": "Quantum yield",
    "model": model_lgb_final_qy,
    "feature_columns": X_qy.columns.tolist(),
}

joblib.dump(artifact_lgb_qy, save_path_qy)
print(f"saved: {save_path_qy}")

saved: models/lgb/artifact_lgb_rdkit_qy.joblib
